In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.regularizers import l1, l2, l1_l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal, GlorotUniform, RandomNormal
import tensorflow as tf
import random
import os
from datetime import datetime

In [19]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.regularizers import l1, l2, l1_l2
from tensorflow.keras.initializers import HeNormal, GlorotUniform, RandomNormal
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# Фиксация случайных состояний для воспроизводимости
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

# === 1. Загрузка и подготовка данных ===
train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values.astype(np.float32)
x_test = test_df[features].values.astype(np.float32)
y_train_full = train_df[target].values.astype(np.float32)
y_test = test_df[target].values.astype(np.float32)

# Нормализация (StandardScaler эквивалентен вашему ручному способу, но надёжнее)
scaler = StandardScaler()
x_train_full = scaler.fit_transform(x_train_full)
x_test = scaler.transform(x_test)

# Разделение на обучение и валидацию (80/20)
val_size = int(0.2 * len(x_train_full))
x_val = x_train_full[:val_size]
y_val = y_train_full[:val_size]
x_train = x_train_full[val_size:]
y_train = y_train_full[val_size:]

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

# === 2. Функция создания модели ===
def create_model(params):
    tf.keras.backend.clear_session()
    model = Sequential()
    
    # Входной слой
    model.add(Dense(
        params['first_layer_units'],
        input_shape=(x_train.shape[1],),
        activation=params['activation'],
        kernel_initializer=params['initializer'],
        kernel_regularizer=params['regularizer']
    ))
    
    if params['batch_norm']:
        model.add(BatchNormalization())
    if params['dropout_rate'] > 0:
        model.add(Dropout(params['dropout_rate']))
    
    # Скрытые слои
    for units in params['hidden_layers']:
        model.add(Dense(
            units,
            activation=params['activation'],
            kernel_initializer=params['initializer'],
            kernel_regularizer=params['regularizer']
        ))
        if params['batch_norm']:
            model.add(BatchNormalization())
        if params['dropout_rate'] > 0:
            model.add(Dropout(params['dropout_rate']))
    
    # Выходной слой
    model.add(Dense(1, activation='linear'))
    return model

# === 3. Обучение модели ===
def train_model(params):
    model = create_model(params)
    
    # Оптимизатор
    lr = params['learning_rate']
    if params['optimizer'] == 'adam':
        optimizer = Adam(learning_rate=lr)
    elif params['optimizer'] == 'sgd':
        optimizer = SGD(learning_rate=lr, momentum=0.9)
    else:  # rmsprop
        optimizer = RMSprop(learning_rate=lr)
    
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    early_stopping = EarlyStopping(
        monitor='val_mae',
        patience=params['patience'],
        restore_best_weights=True,
        verbose=0
    )
    
    history = model.fit(
        x_train, y_train,
        batch_size=params['batch_size'],
        epochs=100,  # увеличено, ранняя остановка сама остановит
        validation_data=(x_val, y_val),
        callbacks=[early_stopping],
        verbose=0
    )
    
    train_mae = model.evaluate(x_train, y_train, verbose=0)[1]
    val_mae = model.evaluate(x_val, y_val, verbose=0)[1]
    test_mae = model.evaluate(x_test, y_test, verbose=0)[1]
    
    return {
        'model': model,
        'train_mae': train_mae,
        'val_mae': val_mae,
        'test_mae': test_mae,
        'actual_epochs': len(history.history['loss']),
        'history': history.history
    }

# === 4. Генерация гиперпараметров (улучшенная стратегия) ===
def generate_params(experiment_id):
    # Архитектура: 1–4 скрытых слоя, умеренный размер
    n_layers = random.randint(1, 4)
    first = random.choice([32, 64, 128])
    hidden = []
    for _ in range(n_layers - 1):
        hidden.append(random.choice([16, 32, 64, 128]))
    
    # Активация
    activation = random.choice(['relu', 'selu', 'elu'])  # tanh исключён — хуже для глубоких сетей
    
    # Оптимизатор и LR
    optimizer = random.choice(['adam', 'sgd', 'rmsprop'])
    if optimizer == 'adam':
        lr = random.choice([1e-4, 5e-4, 1e-3, 2e-3])
    elif optimizer == 'sgd':
        lr = random.choice([1e-3, 5e-3, 1e-2])
    else:  # rmsprop
        lr = random.choice([1e-4, 5e-4, 1e-3])
    
    batch_size = random.choice([32, 64, 128])
    dropout_rate = random.choice([0.0, 0.1, 0.2, 0.3])
    batch_norm = random.choice([True, False])
    
    # Регуляризация весов
    reg_choice = random.choice([None, 'l2_weak', 'l2_strong'])
    if reg_choice == 'l2_weak':
        regularizer = l2(1e-4)
    elif reg_choice == 'l2_strong':
        regularizer = l2(1e-3)
    else:
        regularizer = None
    
    # Инициализатор
    if activation in ('relu', 'elu', 'selu'):
        initializer = HeNormal()
    else:
        initializer = GlorotUniform()
    
    patience = random.choice([8, 12, 15])
    
    return {
        'experiment_id': experiment_id,
        'first_layer_units': first,
        'hidden_layers': hidden,
        'activation': activation,
        'optimizer': optimizer,
        'learning_rate': lr,
        'batch_size': batch_size,
        'dropout_rate': dropout_rate,
        'batch_norm': batch_norm,
        'regularizer': regularizer,
        'initializer': initializer,
        'patience': patience
    }

# === 5. Проведение экспериментов с промежуточным сохранением ===
import os

RESULTS_FILE = 'experiment_results.csv'
BEST_MODEL_FILE = 'best_model.keras'

# Проверка наличия частичных результатов
if os.path.exists(RESULTS_FILE):
    results_df_existing = pd.read_csv(RESULTS_FILE)
    existing_ids = set(results_df_existing['experiment_id'].values)
    start_idx = max(existing_ids) + 1
    results = results_df_existing.to_dict('records')
    print(f"Загружено {len(results)} завершённых экспериментов. Продолжаем с {start_idx}.")
else:
    existing_ids = set()
    start_idx = 0
    results = []

# Инициализация лучшей модели и метрики
best_test_mae = min([r['test_mae'] for r in results], default=float('inf'))
best_model = None
best_params = None

# Если есть сохранённая лучшая модель и она актуальна — загружаем (опционально)
# Здесь для простоты пересоздаём модель при необходимости, но можно и сохранять веса отдельно

TOTAL_EXPERIMENTS = 500
SAVE_EVERY = 10  # Сохранять каждые N экспериментов

print("Starting experiments...")
for i in range(start_idx, TOTAL_EXPERIMENTS):
    if i in existing_ids:
        continue  # Пропускаем уже выполненные

    params_raw = generate_params(i)
    
    # Подготовка параметров для логирования
    reg_name = None
    if params_raw['regularizer'] is not None:
        if isinstance(params_raw['regularizer'], l2):
            if params_raw['regularizer'].l2 == 1e-4:
                reg_name = 'l2(1e-4)'
            elif params_raw['regularizer'].l2 == 1e-3:
                reg_name = 'l2(1e-3)'
    
    try:
        result = train_model(params_raw)
        
        arch_str = str(params_raw['first_layer_units'])
        if params_raw['hidden_layers']:
            arch_str += '-' + '-'.join(map(str, params_raw['hidden_layers']))
        
        log_entry = {
            'experiment_id': i,
            'architecture': arch_str,
            'n_layers': 1 + len(params_raw['hidden_layers']),
            'activation': params_raw['activation'],
            'optimizer': params_raw['optimizer'],
            'learning_rate': params_raw['learning_rate'],
            'batch_size': params_raw['batch_size'],
            'actual_epochs': result['actual_epochs'],
            'dropout_rate': params_raw['dropout_rate'],
            'batch_norm': params_raw['batch_norm'],
            'regularizer': reg_name,
            'initializer': type(params_raw['initializer']).__name__,
            'patience': params_raw['patience'],
            'train_mae': round(result['train_mae'], 4),
            'val_mae': round(result['val_mae'], 4),
            'test_mae': round(result['test_mae'], 4)
        }
        
        results.append(log_entry)
        
        # Обновление лучшей модели
        if result['test_mae'] < best_test_mae:
            best_test_mae = result['test_mae']
            best_model = result['model']
            best_params = log_entry
            # Сохраняем лучшую модель немедленно
            best_model.save(BEST_MODEL_FILE)
            print(f"✅ Новая лучшая модель сохранена (test MAE: {best_test_mae:.4f})")
        
        # Промежуточное сохранение каждые SAVE_EVERY шагов или на последнем
        if (i + 1) % SAVE_EVERY == 0 or i == TOTAL_EXPERIMENTS - 1:
            results_df = pd.DataFrame(results)
            results_df.to_csv(RESULTS_FILE, index=False)
            print(f"💾 Промежуточные результаты сохранены: {len(results)} экспериментов.")
        
        if (i + 1) % 50 == 0:
            print(f"Completed {i + 1}/{TOTAL_EXPERIMENTS}. Best test MAE: {best_test_mae:.4f}")
            
    except Exception as e:
        print(f"Experiment {i} failed: {e}")
        # Даже при ошибке сохраняем текущие результаты
        results_df = pd.DataFrame(results)
        results_df.to_csv(RESULTS_FILE, index=False)
        continue

# === 6. Финальное сохранение (на случай, если SAVE_EVERY не сработало на последнем шаге) ===
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_FILE, index=False)
print("\n✅ Все результаты сохранены в experiment_results.csv")

# Лучшая модель уже сохранялась по ходу, но если она не была обновлена — проверим
if best_model is not None and not os.path.exists(BEST_MODEL_FILE):
    best_model.save(BEST_MODEL_FILE)
    print(f"✅ Лучшая модель сохранена (test MAE: {best_test_mae:.4f})")

# === 7. Анализ результатов ===
print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ АНАЛИЗА")
print("="*60)
print(f"Всего моделей: {len(results)}")
print(f"Лучший test MAE: {best_test_mae:.4f}")
if best_params:
    print(f"Лучшая архитектура: {best_params['architecture']}")
    print(f"Активация: {best_params['activation']}, Оптимизатор: {best_params['optimizer']}")
    print(f"Регуляризация: {best_params['regularizer']}, BatchNorm: {best_params['batch_norm']}")
    print(f"Dropout: {best_params['dropout_rate']}, Batch size: {best_params['batch_size']}")

# Топ-10
top10 = results_df.nsmallest(10, 'test_mae')
print("\nТоп-10 моделей по test MAE:")
print(top10[['experiment_id', 'architecture', 'activation', 'test_mae']].to_string(index=False))

Train: (13600, 8), Val: (3400, 8), Test: (3000, 8)
Загружено 4 завершённых экспериментов. Продолжаем с 5.
Starting experiments...
✅ Новая лучшая модель сохранена (test MAE: 0.0046)


c:\Users\dpomi\source\repos\data_science\.venv310\lib\site-packages\keras\initializers\initializers_v2.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Experiment 7 failed: Graph execution error:

Detected at node 'gradient_tape/mean_squared_error/Reshape' defined at (most recent call last):
    File "C:\Users\dpomi\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "C:\Users\dpomi\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "c:\Users\dpomi\source\repos\data_science\.venv310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "c:\Users\dpomi\source\repos\data_science\.venv310\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "c:\Users\dpomi\source\repos\data_science\.venv310\lib\site-packages\ipykernel\kernelapp.py", line 758, in start
      self.io_loop.start()
    File "c:\Users\dpomi\source\repos\data_science\.venv310\lib\site-packages\tornado\platform\asynci

KeyboardInterrupt: 